**[Team Member Names], Team [XX##]**

## EE 242 Lab 2b – Convolution - Denoising audio signals

In this lab, you will work as a team through a series of exercises to introduce you to working with audio signals and explore the impact of convolution and smoothing on the sound of the signals.

**Submission Format:** PDF

Please refer to Lab0 for how to convert your notebook to a PDF.

### Import Cell

We'll refer to the below cell as the "import cell." Every module you import should be imported in it. We've added some for you to start with.

In [ ]:
from math import tau  # Note: tau = 2 * pi, tau FTW! #team_tau

import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import scipy.io.wavfile as wav
import scipy.signal as sig
from matplotlib.ticker import MultipleLocator

## Assignment 2 -- Smoothing Signals

In this assignment, we’ll implement a moving window smoothing function to show how you can use convolution to remove noise from a signal. The noise signal is generated randomly, so you can run the cell multiple times to see how the results look for different signals. 

1. Complete the starter code provided by using [numpy.random.Generator.normal()](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.normal.html) to generate normal noise of length `N` and scale `noise_scale`. 

    Set your `seed` value to be your team number. Then, use `generate_random_signal()` to create 
    an array called `noisy_signal` that is equal to `base_signal + noise`. Finally, create a 
    single plot containing both `base_signal` and `noisy_signal` against `t` with the 
    following specifications:

    - Plot `noisy_signal` using matplotlib's default first color.
    
    - Plot `base_signal` as a black dashed line with linewidth 1 

    - Ensure that `base_signal` is plotted on top of `noisy_signal`.

    - Legend labels:
        - Base signal: `base signal`

        - Noisy signal: `noisy signal`

Tips:
- Plot `base_signal` second so that it is on the top layer.

- You can set the linewidth and formatting using settings of [`matplotlib.pyplot.plot()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html)

- You add a legend using [`matplotlib.pyplot.legend()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.legend.html)

- Legend labeling can be controlled using the `label` parameter of [`matplotlib.pyplot.plot()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html)


Note: You may see that older code uses `numpy.random.normal()` instead of `numpy.random.Generator.normal()`. NumPy now recommends using the newer `numpy.random.Generator.normal()`, typically accessed by first creating a random number generator using `numpy.random.default_rng()`. The legacy method should be avoided in new code.


In [ ]:
# Set me to your team number!
seed = 0

In [ ]:
def generate_random_signal(
    seed: int, 
    fs: float, 
    t_final: float, 
    noise_scale: float = 1
) -> tuple[
    np.ndarray[tuple[int], np.dtype[np.float64]],
    np.ndarray[tuple[int], np.dtype[np.float64]],
    np.ndarray[tuple[int], np.dtype[np.float64]],
]:
    """Generate a random signal with the given sampling rate from 0 to `t_final`.

    Parameters
    ----------
    seed : int
        Random seed value for random number generation. The same seed will always in the
        same signal being generated.

    fs : float
        The sampling rate in [Hz].

    t_final : float
        The duration of the signal in [s].

    noise_amplitude : float
        The amplitude of the noise. 

    Returns
    -------
    t : ndarray[tuple[int], numpy.datatype[numpy.float64]]
        Time vector associated with base_signal.

    base_signal : ndarray[tuple[int], numpy.datatype[numpy.float64]]
        Randomly generated base signal without noise.

    noise : ndarray[tuple[int], numpy.datatype[numpy.float64]]
        Randomly generated noise vector.
    """
    rng = np.random.default_rng(seed)
    N_pieces = 10
    N = int(t_final * fs) + 1
    amplitude = 20

    t = np.linspace(0, t_final, num=N)
    base_signal = np.interp(
        np.linspace(0, N_pieces, N), 
        np.arange(0, N_pieces), 
        rng.random(N_pieces) * amplitude
    )


    # Generate guassian (normal) random noise
    noise = _

    return t, base_signal, noise


fs = 25
t_final = 10
noise_amplitude = 0.25

t, base_signal, noise = generate_random_signal(seed, fs, t_final, noise_amplitude)

noisy_signal = _

# Your plots here:


2. We can create a smoothed version of the signal called `filtered_signal_manual` by replacing every point with the average of the $M = 12$ nearest neighbors on *each side* (i.e., a centered moving average with a total window size of $2M + 1$ points) via `numpy.mean()`. For this exercise only, we'd like you to manually compute the filter using a Python `for` loop rather than `numpy.convolve()` to get a better feel for what the process of implementing convolution in real-time systems, such as on embedded hardware.

    To get started, create an array of the same length as `noisy_signal` to store the result in. 

    Tips:
    - Use `numpy.zeros_like()` or `numpy.empty_like()` to create an array that is the same length as another array.

    -  To get the `N` points centered around the `kth` sample use NumPy slicing.

    - You will need to make a decision as to how to handle the first and last k samples, for which there won’t be a full k samples available in both directions. In a single plot, plot the noisy signal and the filtered signal overlaid on the original signal.

In [ ]:
# Your code here

3. Recall from Lab 2a that convolving with the impulse response $[1, \dots, 1]$ is the $N$ point moving sum. If we simply divide by the number of points, $N$, we get the $N$ point moving average: $[\frac{1}{N}, \dots, \frac{1}{N}]$. Use this formula to create $N$ point moving average impulse responses for $N = 2, 5, 10, 25$, called `h_2pt_avg`, `h_5pt_avg`, `h_10pt_avg`, `h_25pt_avg`, respectively. Convolve each of these impulse responses using `numpy.convolve()` and `mode = 'full'` with `noisy_signal` to generate `filt_signal_2pt`, `filt_signal_5pt`, `filt_signal_10pt`, and `filt_signal_25pt`. Create associated time vectors, `t_2pt`, `t_5pt`, `t_10pt`, `t_25pt`. Finally, create a figure to compare the results of each filter as well as `filt_signal_manual` with the following specifications:
    - Layout: 5 rows, 1 column
        - Plot in order of increasing number of points in the filter, `N`. Then plot `filt_signal_manual` at the bottom.

    - Signal formatting:
        - Base Signal: Plot the base-signal as black dashed line with `linewidth = 1`. Ensure it is layered *over* the noisy signal and the filtered signal.

        - Noisy Signal: Plot the noisy signal using matplotlib's default first color.

        - Filtered Signal: Plot the filtered signal using matplotlib's default second color.

    - X-axis limits: $[0, 10]$, with shared labeling
    
    - Y-axis limits: $[-2, 2]$, shared limits

    - Y-axis minor ticks: Minor tick interval of $0.5$

    - Labels and titles: Title each plot `[N] point moving average`, where `N` is the number of points in the moving average filter.

    - Legend: Centered, below the final plot

    - Legend labels:
        - Base signal: `base signal`

        - Noisy signal: `noisy signal`
        
        - Filtered signal: `filtered signal` 
    
    - Spacing: Make sure titles do not overlap with the graph
    
    - Grid: No gridlines (this is the default)

Tips:

- The time points associated with each signal, including `filt_signal_manual` is simply just the sample index divided by the sampling frequency.

- You add a legend using [`matplotlib.pyplot.legend()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.legend.html)

- Legend labeling can be controlled using the `label` parameter of [`matplotlib.pyplot.plot()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html)

- You can control the positioning of the legend by using the `bbox_to_anchor` argument to [`matplotlib.pyplot.legend()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.legend.html)

- You can control the y axis tick interval by using [`matplotlib.axis.Axis.set_minor_locator()`](https://matplotlib.org/stable/gallery/ticks/major_minor_demo.html) and [`matplotlib.ticker.MultipleLocator()`](https://matplotlib.org/stable/api/ticker_api.html#matplotlib.ticker.MultipleLocator)


In [ ]:
h_2pt_avg = _
h_5pt_avg = _
h_10pt_avg = _
h_25pt_avg = _

filt_signal_2pt = _
t_2pt = _

filt_signal_5pt = _
t_5pt = _

filt_signal_10pt = _
t_10pt = _

filt_signal_25pt = _
t_25pt = _

# Add plots here:


###  Discussion

1. In Assignment 2 Part 2, you applied moving average filters of different lengths ($N = 2, 5, 10, 25$) to a noisy signal. Describe the trade-off you observed: as $N$ increases, what happens to the noise reduction versus the ability to track rapid changes in the base signal?

[Your answer here]

2. The moving average filter is a *low-pass filter*. Based on your observations, why might it be effective at reducing noise? What types of signals would be poorly suited for noise reduction via a moving average filter?

[Your answer here]

3. Can you explain what causes the filtering artifacts at the edges of the filtered signal? What kind of considerations would you need to make when implementing your own convolution for a real-time signal processing applications (e.g., on an embedded system) because of this?

    Hint: Read [`numpy.convolve()`](https://numpy.org/doc/stable/reference/generated/numpy.convolve.html#numpy-convolve) and consider Lab2a Part 5.

[Your answer here]

4. For larger values of $N$, you should see that the filtered signal lags behind the original signal. The number of samples of delay introduced is called *group delay*. It is possible to compensate for this group delay by shifting the filtered signal, either manually or by using `mode = 'same'` with `numpy.convolve()`. Try it yourself to see.

    a. What causes the group delay? Can you explain why the lag seems to increase when we increase the value of $N$, the number of points in the moving average?
    
    b. How many samples do you need to advance the filtered signal by to compensate for the group delay introduced by an $N$ point filter? Why? Your answer does not need to provide rigorous mathematical justification. Rough intuition is acceptable.
    
    c. Given this group delay, what kinds of considerations would be important for real-time signal processing applications?

    Hints:
    - (b): The moving average filter replaces every value $a_k$ with a weighted average of $a_k$ and the $N - 1$ preceding points. Think about what it would mean to take a mean of $N$ points before and including $a_k$ in the filter. If it helps, consider a concrete example with the 3 point moving average $[\frac{1}{3}, \frac{1}{3}, \frac{1}{3}]$, which replaces $a_k$ with $\frac{1}{3}a_{k - 2} + \frac{1}{3}a_{k - 1} + \frac{1}{3}a_k$. Now consider the sequence $[\dots, a_{k - 1}, a_k, a_{k + 1}, \dots]$. We can think of a point (roughly) as being "represented" by the point in the filter where it is *centered*. This looks like the filter window being shifted so it is centered over a point. This means the point that replaces $a_k$ cannot be the point which represents it since it is not centered in the mean sum. What does a mean centered around $a_k$ look like? How does one need to shift the filter to create the point that represents $a_k$? Try considering also filters of size $5$ and $7$. Even numbers of points can't center perfectly so it's fine not to consider them (the answer turns out to be the same either way).

    - (c): When we manually shift the signal to recenter the output, we are advancing the filtered signal so that future values of the convolution appear sooner than they would normally.

[Your answer here]

## Assignment 3 -- Removing Noise from an Audio Signal

In this assignment, we’ll artificially add noise to an audio signal and then apply the moving average filter you used in Assignment 2 to remove it.

1. First, read in the trombone sound file provided into the array `tr_orig`.  This is a mono signal, so we will only need to worry about the single channel. Create a *white noise* sequence that is the same length of `tr_orig` that has the range: $[-100, 100]$ and name this noise sequence `tr_noise`. Finally, add the two signals to the array `tr_noisy` and save it as a new `.wav` file called `trombone11_noisy.wav`. Finally, play both `tr_orig` and `tr_noisy` and have your TA listen to them.

Tips:
- Start by creating a new random number generator with [`numpy.random.default_rng()`](https://numpy.org/doc/stable/reference/random/generator.html) using the same seed you used in Assignment 2.

- White noise can be thought of as a signal where each sample is drawn from an independent random variable. There are many valid distributions. We saw normal white noise in Assignment 2, and in this problem we will explore uniform white noise, where each sample is *uniformly distributed* (i.e., each number in the distribution has an equal chance of being chosen). We can generate uniform random number between 0 and 1 using [`numpy.random.Generator.random()`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.random.html#numpy.random.Generator.random). Check the documentation for details on how we can transform this to a range of $[-100, 100]$.

2. We can create a smoothed version of the signal called `filtered_signal_manual` by replacing every point with the average of the $M = 20$ nearest neighbors on *each side* (i.e., a centered moving average with a total window size of $2M + 1$ points) via `numpy.mean()`. 

    To get started, create an array of the same length as `noisy_signal` to store the result in. 

    Tips:
    - Use `numpy.zeros_like()` or `numpy.empty_like()` to create an array that is the same length as another array.

    -  To get the `N` points centered around the `kth` sample use NumPy slicing.

    - You will need to make a decision as to how to handle the first and last k samples, for which there won’t be a full k samples available in both directions. In a single plot, plot the noisy signal and the filtered signal overlaid on the original signal.

In [ ]:
# Create random number generator:
rng = _

# Read in trombone11.wav
fs, tr_orig = _

# Generate noise and create noisy signal
tr_noise = _
tr_noisy = _

# Write noisy signal to file.


2. Now create an $N$ point moving average filter called `h` to remove the noise we've generated. Experiment with different values of $N$ to get a good result. Call the filtered signal `tr_filt` and save it to a new `.wav` file called `trombone11_fil.wav`. Then create time vectors `t_orig` and `t_filt` for `tr_orig` and `tr_filt`, respectively. Plot both `tr_orig` and `tr_filt` over their respective time vectors on the same plot with the following specifications:
- `tr_orig`: Matplotlib's default first color
- `tr_filt`: Matplotlib's default second color, plotted *over* `tr_orig` with `alpha = 0.5`
- Legend: Label `tr_orig` as `original` and `tr_filt` as `filtered`.

Tips:
- The resulting filtered signal will be attenuated. You may want to add a constant gain by multiplying the filtered signal by some constant to offset this.

In [ ]:
# Experiment with different values of N
N = _

# Create a moving average filter based on the value of N
h = _

# Filter the signal by convolving h with tr_noisy
tr_filt = _

# Generate time vectors
t_orig = _
t_filt = _

# Generate plots:

plt.plot(t_orig, tr_orig, label="original")
plt.plot(t_filt, tr_filt, alpha=0.5, label="filtered")
plt.legend()
plt.show()

# Write tr_filt to trombone11_filt.wav:

# Play tr_filt:


###  Discussion

1. What does the white noise sound like when added to the trombone file?

[Your answer here]

2. How does the value of $N$ affect the filtered signal?

[Your answer here]